# Required Skills Of An AI Engineer

Run `4-classify-job-postings.ipynb` first so that `Job Posting Data/2-llm_filtered_jobs.jsonl` exists.

This notebook uses an LLM to extract a normalized set of AI-engineering skills for each accepted job posting, saves the per-job results to `Job Posting Data/3-extracted_skills.jsonl`, and then calculates how often each skill appears across the postings.


In [7]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import display

llm_filtered_jsonl_path = Path("Job Posting Data") / "2-llm_filtered_jobs.jsonl"
skills_jsonl_path = Path("Job Posting Data") / "3-extracted_skills.jsonl"

if not llm_filtered_jsonl_path.exists():
    raise FileNotFoundError(
        f"LLM-filtered jobs JSONL file not found: {llm_filtered_jsonl_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

jobs = pd.read_json(llm_filtered_jsonl_path, lines=True)

if jobs.empty:
    raise ValueError("The LLM-filtered jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first.")

if "description" not in jobs.columns:
    raise KeyError("The LLM-filtered jobs JSONL file must contain a 'description' column.")

jobs = jobs[jobs["description"].notna()].copy()
jobs["description"] = jobs["description"].astype(str).str.strip()
jobs = jobs[jobs["description"] != ""].copy()
jobs = jobs.reset_index(drop=True)

if jobs.empty:
    raise ValueError("The LLM-filtered jobs JSONL file contains no usable job descriptions for skill extraction.")

skill_categories = [
    "ai-engineering",
    "ml",
    "frontend",
    "backend",
    "databases",
    "data",
    "cloud",
    "ops",
    "languages",
    "other",
]

client = OpenAI()
max_description_chars = 6000
max_skills_per_job = 15
skill_results = []

instructions = """
You extract required skills from AI engineering job postings.

Extract the concrete required skills from each job posting.
Use the provided categories, but do not limit yourself to any predefined skill list.
Include a skill only when it is clearly required, strongly implied, or a central part of the role.
Do not include a skill just because the posting vaguely mentions AI.
Use concise normalized skill names like 'python', 'rag', 'prompt engineering', 'aws', 'docker', 'kubernetes', 'sql', 'langchain'.
Do not use category names themselves as skills unless the posting truly names that exact thing as a skill.
Return only the skills that are actually relevant for the posting.
Return at most 15 skills.
""".strip()

skill_schema = {
    "type": "object",
    "properties": {
        "skills": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string", "minLength": 1, "maxLength": 60},
                    "category": {"type": "string", "enum": skill_categories}
                },
                "required": ["skill", "category"],
                "additionalProperties": False
            },
            "maxItems": 15
        }
    },
    "required": ["skills"],
    "additionalProperties": False,
}

category_text = "\n".join(f"- {category}" for category in skill_categories)

for i, (_, job) in enumerate(jobs.iterrows(), start=1):
    title = "" if pd.isna(job.get("title")) else str(job.get("title")).strip()
    company = "" if pd.isna(job.get("company")) else str(job.get("company")).strip()
    job_url = "" if pd.isna(job.get("job_url")) else str(job.get("job_url")).strip()
    description = str(job.get("description", "")).strip()

    description_for_prompt = description[:max_description_chars]
    if len(description) > max_description_chars:
        description_for_prompt += "\n\n[Description truncated for skill extraction.]"

    print(f"Extracting skills for job {i}/{len(jobs)}: {title} @ {company}")

    prompt = f"""Extract the required skills for this job posting.\n\nUse only these categories:\n{category_text}\n\nReturn concrete skills and assign each skill to one category. Return at most {max_skills_per_job} skills.\n\nTitle: {title}\nCompany: {company}\nURL: {job_url}\n\nDescription:\n{description_for_prompt}"""

    result = None
    last_error = None

    for max_output_tokens in (400, 800, 1200):
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=instructions,
            input=prompt,
            max_output_tokens=max_output_tokens,
            text={
                "format": {
                    "type": "json_schema",
                    "name": "ai_engineering_skill_extraction",
                    "schema": skill_schema,
                    "strict": True,
                },
                "verbosity": "low",
            },
        )

        try:
            result = json.loads(response.output_text)
            break
        except json.JSONDecodeError as exc:
            last_error = exc
            if response.incomplete_details is None and response.status == "completed":
                continue

    if result is None:
        raise ValueError(
            f"Failed to parse structured skill output for '{title}' at '{company}'. Last error: {last_error}"
        )
    extracted_skills = []
    seen_pairs = set()

    for item in result.get("skills", []):
        raw_skill = str(item.get("skill", "")).strip()
        category = str(item.get("category", "")).strip().lower()

        if not raw_skill or category not in skill_categories:
            continue

        normalized_skill = raw_skill.lower()
        pair = (normalized_skill, category)
        if pair in seen_pairs:
            continue

        seen_pairs.add(pair)
        extracted_skills.append(
            {
                "skill": normalized_skill,
                "category": category,
            }
        )

    extracted_skills = sorted(extracted_skills, key=lambda item: (item["category"], item["skill"]))
    extracted_skill_labels = [f"{item['skill']} [{item['category']}]" for item in extracted_skills]

    skill_results.append(
        {
            "title": title,
            "company": company,
            "job_url": job_url,
            "extracted_skills": extracted_skills,
            "extracted_skill_labels": extracted_skill_labels,
        }
    )

skills_by_job = pd.DataFrame(skill_results)
skills_by_job.to_json(skills_jsonl_path, orient='records', lines=True, force_ascii=False)

print(f"Saved extracted skills to: {skills_jsonl_path.resolve()}")
skills_by_job_to_show = skills_by_job[["title", "company", "extracted_skill_labels"]].copy()
skills_by_job_to_show.index = range(1, len(skills_by_job_to_show) + 1)
display(skills_by_job_to_show)


Extracting skills for job 1/40: Principal AI Engineer @ VideoAmp
Extracting skills for job 2/40: AI Engineer / Agentic and Generative AI @ Realign
Extracting skills for job 3/40: AI Engineer, Enterprise AI (Principal/Sr. Principal/Distinguished) @ Palo Alto Networks
Extracting skills for job 4/40: Junior Applied AI Engineer- HRIS @ Fortinet
Extracting skills for job 5/40: Principal AI Engineer @ Burrell Behavioral Health
Extracting skills for job 6/40: Principal, Software Engineer - GenAi @ Walmart
Extracting skills for job 7/40: Gen AI Engineer @ Persistent Systems
Extracting skills for job 8/40: Senior Gen AI Engineer (USC/GC) W2 Role @ Hudson Manpower
Extracting skills for job 9/40: Senior Artificial Intelligence (AI) Engineer (SWE-3) @ Leidos
Extracting skills for job 10/40: Senior AI/ML Engineer @ Quest Global
Extracting skills for job 11/40: Principal Engineer - Agentic AI (US) @ TD
Extracting skills for job 12/40: CrowdStrike AI Security Engineer @ Openkyber
Extracting skills fo

,title,company,extracted_skill_labels
1,Principal AI Engineer,VideoAmp,"[agentic workflows [ai-engineering], evaluatio..."
2,AI Engineer / Agentic and Generative AI,Realign,"[agentic ai [ai-engineering], evaluation frame..."
3,"AI Engineer, Enterprise AI (Principal/Sr. Prin...",Palo Alto Networks,"[agentic ai [ai-engineering], ai observability..."
4,Junior Applied AI Engineer- HRIS,Fortinet,"[chatbots [ai-engineering], llms [ai-engineeri..."
5,Principal AI Engineer,Burrell Behavioral Health,"[context management [ai-engineering], groundin..."
6,"Principal, Software Engineer - GenAi",Walmart,"[agents [ai-engineering], genai [ai-engineerin..."
7,Gen AI Engineer,Persistent Systems,"[genai architecture [ai-engineering], langchai..."
8,Senior Gen AI Engineer (USC/GC) W2 Role,Hudson Manpower,"[crewai [ai-engineering], generative ai [ai-en..."
9,Senior Artificial Intelligence (AI) Engineer (...,Leidos,"[hugging face [ai-engineering], openai api [ai..."
10,Senior AI/ML Engineer,Quest Global,"[agentic ai [ai-engineering], genai [ai-engine..."


In [8]:
skills_by_job = pd.read_json(skills_jsonl_path, lines=True)

if skills_by_job.empty:
    raise ValueError("No extracted skills were found. Run the previous cell first.")

total_jobs = len(skills_by_job)
exploded = skills_by_job.explode("extracted_skills").copy()
exploded = exploded[exploded["extracted_skills"].notna()].copy()

if exploded.empty:
    raise ValueError("The extracted skills file does not contain any skills. Run the previous cell again.")

exploded["skill"] = exploded["extracted_skills"].apply(lambda item: item["skill"])
exploded["category"] = exploded["extracted_skills"].apply(lambda item: item["category"])
category_counts = []
skill_counts = []

for category in skill_categories:
    count = int(skills_by_job["extracted_skills"].apply(lambda skills: any(item["category"] == category for item in skills)).sum())
    percentage = round((count / total_jobs) * 100, 1)
    category_counts.append(
        {
            "category": category,
            "jobs_mentioning_category": count,
            "total_jobs": total_jobs,
            "percentage": percentage,
        }
    )

for (category, skill_name), skill_frame in exploded.groupby(["category", "skill"]):
    count = int(skill_frame.index.nunique())
    percentage = round((count / total_jobs) * 100, 1)
    skill_counts.append(
        {
            "category": category,
            "skill": skill_name,
            "jobs_mentioning_skill": count,
            "total_jobs": total_jobs,
            "percentage": percentage,
        }
    )

category_stats = pd.DataFrame(category_counts).sort_values(
    by=["jobs_mentioning_category", "category"], ascending=[False, True]
).reset_index(drop=True)

skill_stats = pd.DataFrame(skill_counts).sort_values(
    by=["jobs_mentioning_skill", "category", "skill"], ascending=[False, True, True]
).reset_index(drop=True)

category_stats["share_of_jobs"] = category_stats.apply(
    lambda row: f"{row['jobs_mentioning_category']}/{row['total_jobs']} ({row['percentage']}%)",
    axis=1,
)

skill_stats["share_of_jobs"] = skill_stats.apply(
    lambda row: f"{row['jobs_mentioning_skill']}/{row['total_jobs']} ({row['percentage']}%)",
    axis=1,
)

category_stats_to_show = category_stats[["category", "share_of_jobs"]].copy()
category_stats_to_show.index = range(1, len(category_stats_to_show) + 1)

top_25_skill_stats = skill_stats[["category", "skill", "share_of_jobs"]].head(25).copy()
top_25_skill_stats.index = range(1, len(top_25_skill_stats) + 1)

print(f"Top 25 skills out of {len(skill_stats)} extracted skills")
display(category_stats_to_show)
display(top_25_skill_stats)

for category in category_stats["category"]:
    category_top_10 = skill_stats[skill_stats["category"] == category][["skill", "share_of_jobs"]].head(10).copy()
    category_top_10.index = range(1, len(category_top_10) + 1)
    print(f"Top 10 skills for category: {category}")
    display(category_top_10)


Top 25 skills out of 305 extracted skills


,category,share_of_jobs
1,ai-engineering,40/40 (100.0%)
2,languages,33/40 (82.5%)
3,backend,30/40 (75.0%)
4,ops,25/40 (62.5%)
5,cloud,22/40 (55.0%)
6,ml,21/40 (52.5%)
7,databases,19/40 (47.5%)
8,data,16/40 (40.0%)
9,other,14/40 (35.0%)
10,frontend,10/40 (25.0%)


,category,skill,share_of_jobs
1,languages,python,32/40 (80.0%)
2,ai-engineering,prompt engineering,20/40 (50.0%)
3,ai-engineering,rag,17/40 (42.5%)
4,ai-engineering,langchain,14/40 (35.0%)
5,cloud,aws,12/40 (30.0%)
6,ai-engineering,llms,11/40 (27.5%)
7,ai-engineering,agentic ai,9/40 (22.5%)
8,ops,docker,9/40 (22.5%)
9,databases,vector databases,8/40 (20.0%)
10,languages,typescript,7/40 (17.5%)


Top 10 skills for category: ai-engineering


,skill,share_of_jobs
1,prompt engineering,20/40 (50.0%)
2,rag,17/40 (42.5%)
3,langchain,14/40 (35.0%)
4,llms,11/40 (27.5%)
5,agentic ai,9/40 (22.5%)
6,langgraph,5/40 (12.5%)
7,evaluation frameworks,4/40 (10.0%)
8,large language models,4/40 (10.0%)
9,mcp,4/40 (10.0%)
10,agentic systems,3/40 (7.5%)


...

,skill,share_of_jobs
1,data pipelines,6/40 (15.0%)
2,data engineering,4/40 (10.0%)
3,analytics,2/40 (5.0%)
4,data governance,2/40 (5.0%)
5,a/b testing,1/40 (2.5%)
6,airflow,1/40 (2.5%)
7,bi tools,1/40 (2.5%)
8,big data,1/40 (2.5%)
9,clinical trial data,1/40 (2.5%)
10,data chunking,1/40 (2.5%)


Top 10 skills for category: other


,skill,share_of_jobs
1,technical leadership,3/40 (7.5%)
2,technical communication,2/40 (5.0%)
3,advertising domain experience,1/40 (2.5%)
4,computer science,1/40 (2.5%)
5,consulting,1/40 (2.5%)
6,cross-functional collaboration,1/40 (2.5%)
7,customer-facing engineering,1/40 (2.5%)
8,debugging,1/40 (2.5%)
9,excel automation,1/40 (2.5%)
10,finance automation,1/40 (2.5%)


Top 10 skills for category: frontend


,skill,share_of_jobs
1,react,6/40 (15.0%)
2,dashboarding,1/40 (2.5%)
3,htmx,1/40 (2.5%)
4,next.js,1/40 (2.5%)
5,react.js,1/40 (2.5%)
6,typescript,1/40 (2.5%)
